# Statistics of Li et al. 2024 (Nat. Mach. Intell.) on our runs
Physical statistics of the paper, single component (`codes/paper_figures_li.py`): increment PDFs (Fig 1a), acceleration PDF and correlation (Fig 2), $S_p$ (Fig 4a, + $p=8$), $F^{(p)}$ (Fig 4b), $\zeta(p,\tau)$ (Fig 5), $E$, $A$ (Table 2). Run on Jean Zay (needs `saved_results/`).

Bands as in the paper: min/max over 10 batches of trajectories (Fig 5: 30 for the data, 10 for the synthesis), data and synthesis alike. Data dotted, synthesis solid.

| name | label | what |
|---|---|---|
| B | `turbulence_nested_M3_bicw_allmass` | latest scalar potential (constrained windows + all masses), no Scat4 |
| with Scat4 | `turbulence_ggd_mass_final` | previous scalar version (tail-mass fix) + Scat4 |

Units: these runs coarse-grain once (`subseries_len` 512 → `target_len` 256), so one sample = 2 $dt_s$ ≈ 0.21 $\tau_\eta$ and $\tau_\eta$ = 4.67 samples. The smallest lag is ≈ 0.21 $\tau_\eta$ (paper: 0.1), the largest 254 samples ≈ 54 $\tau_\eta$: the $\tau/\tau_\eta = 100$ PDF of Fig 1a cannot be computed and is skipped. The windows are periodized, so the large-$\tau$ end (S₂ peaks near T/2 ≈ 27 $\tau_\eta$) is not comparable to the paper's raw trajectories.

In [ ]:
%matplotlib inline
%load_ext autoreload
%autoreload 2
import matplotlib.pyplot as plt
from run_comparison import load_runs, reference_data
from paper_figures_li import *

In [ ]:
RUNS = {   # {display name: --label}
    'B: constrained + all masses': 'turbulence_nested_M3_bicw_allmass',
    'with Scat4 (mass_final)': 'turbulence_ggd_mass_final',
}
SEED = 900
SAVE_DIR = None   # e.g. 'figures/li2024' to save pdfs instead of displaying
runs, missing = load_runs(RUNS, seed=SEED)
assert not missing, missing
steps = {k: r['cfg']['subseries_len'] // r['cfg']['target_len'] for k, r in runs.items()}
assert len(set(steps.values())) == 1, steps          # same coarse-graining -> one x1 for all runs
STEP = next(iter(steps.values()))
cfg0 = next(iter(runs.values()))['cfg']
x1 = reference_data(cfg0)
xs = {k: r['xt'] for k, r in runs.items()}
TAU_ETA_RUN, MAX_LAG_RUN = TAU_ETA / STEP, MAX_LAG // STEP
print({k: tuple(v.shape) for k, v in xs.items()}, 'data', tuple(x1.shape))
print(f'step {STEP}: tau_eta = {TAU_ETA_RUN:.2f} samples, max_lag = {MAX_LAG_RUN}')

## Fig 1a: standardized PDFs of $\delta_\tau V$ at $\tau/\tau_\eta$ = 1, 2, 5 (100 skipped)

In [ ]:
for name, x in xs.items():
    print(name)
    panel_li_fig1a_increment_pdfs(x1, x, label=name, tau_eta=TAU_ETA_RUN, save_dir=SAVE_DIR)

## Fig 2: acceleration PDF (one-step increment of the coarse-grained signal, 2 $dt_s$)

In [ ]:
for name, x in xs.items():
    print(name)
    panel_li_fig2_acceleration_pdf(x1, x, label=name, save_dir=SAVE_DIR)

## Fig 2 inset: acceleration correlation $C_\tau = \langle a(t+\tau)\,a(t)\rangle$

In [ ]:
for name, x in xs.items():
    print(name)
    panel_li_fig2_inset_acceleration_correlation(x1, x, label=name, max_lag=MAX_LAG_RUN, tau_eta=TAU_ETA_RUN, save_dir=SAVE_DIR)

## Fig 4a: structure functions $S_p$, $p$ = 2, 4, 6, 8 (alpha 1 → 0.4)

In [ ]:
for name, x in xs.items():
    print(name)
    panel_li_fig4a_structure_functions(x1, x, label=name, max_lag=MAX_LAG_RUN, tau_eta=TAU_ETA_RUN, save_dir=SAVE_DIR)

## Fig 4b: generalized flatness $F^{(p)}$, $p$ = 4, 6, 8

In [ ]:
for name, x in xs.items():
    print(name)
    panel_li_fig4b_flatness(x1, x, label=name, max_lag=MAX_LAG_RUN, tau_eta=TAU_ETA_RUN, save_dir=SAVE_DIR)

## Fig 5: local scaling exponent $\zeta(p,\tau)$, $p$ = 4, 6, 8 (dotted: non-intermittent $p/2$)

In [ ]:
for p in (4, 6, 8):
    for name, x in xs.items():
        print(name, 'p =', p)
        panel_li_fig5_local_exponent(x1, x, label=name, p=p, max_lag=MAX_LAG_RUN, tau_eta=TAU_ETA_RUN, save_dir=SAVE_DIR)

## Table 2: $E = \langle V^2\rangle$, $A = \langle a^2\rangle$ (x1 is standardized: not the paper's physical units)

In [ ]:
for name, x in xs.items():
    print(name)
    li_table2(x1, x)